---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-43: Creating, Deploying and Accessing your own MCP Servers</h1>

# Learning agenda of this notebook

1. Design and Build your Own MCP Server
2. Run Your MCP Server Locally in Debugging/Developer Mode and test it via MCP Inspector
3. Run your MCP Server Locally in Production Mode and Access it in Claude Desktop
4. Deploy your MCP Server on render.com and Access it in Claude Desktop
5. Access a Remote Flight-Search MCP Server from Claude Desktop (Custom Connector or Config File + `mcp-remote`)
6. Access a Remote Hotel-Search MCP Server from Claude Desktop (Custom Connector or Config File + `mcp-remote`)

<font color=purple> <h1> 1. Design and Build Your Own MCP Server</h1>

<div style="text-align:center;">
    <img src="../images/mcp_demo_server_figure.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

### Option 1: Server Class (Low-Level MCP)
- Gives you full control over everything: registering tools, defining schemas, wiring logic, startup, and error handling.
- Best for custom or very specific use cases where every detail matters.
- But it’s verbose and prone to mistakes, because you have to write a lot of boilerplate code.
- Think of it like programming in C/C++: very powerful, but less forgiving.
### Option 2: FastMCP Class: (Higher-level abstraction)
- Built on top of the raw server logic, it handles most of the setup automatically.
- Automatically registers tools, infers input/output types from your functions, sets up the server, and manages client interactions.
- You just write functions and decorate them with `@mcp.tool`.
- Supports almost all MCP features and is actively maintained by the community.
- Think of it like programming in Python: simpler, faster, and more user-friendly than low-level code.</font>

## Step 1. Setting up the Environment with uv and git

**i. Creating a project from scratch**
```bash
mkdir mcp_demo_server                # Create a new project directory
cd mcp_demo_server                   # Navigate into the directory
uv init . --no-package --python 3.12 # Initialize the project (flat layout with main.py), pinned to Python 3.12
ls -a
.	..	.git	.gitignore	.python-version	main.py	pyproject.toml	README.md
```

- `uv init` creates `pyproject.toml` (the project manifest), `main.py`, `README.md`, and a `.python-version` file recording the interpreter version. It also initializes a git repository with a `.gitignore` (unless the folder is already inside a git repository), so there is no need to run `git init` yourself.
> **Why `--no-package`?** Since **uv 0.12** (July 2026), a plain `uv init` creates a *packaged* project: the code lives in `src/<project_name>/__init__.py`, there is **no** `main.py`, and `pyproject.toml` gets a `[build-system]` section. `--no-package` gives the flat layout with a root-level `main.py`, which is the right shape for scripts and servers like ours. On uv versions older than 0.12 this flag is simply the default behaviour, so the command works everywhere.

**ii. Inspecting the manifest**
- Note that the dependencies list is empty in the beginning, as we have not installed any packages yet:
```bash
cat pyproject.toml
[project]
name = "mcp-demo-server"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.12"
dependencies = []
```
- Depending on your git settings, uv may also add an `authors = [...]` entry.



**iii. Adding the dependencies**
- Our server needs `fastmcp`, `requests` (for the weather API) and `python-dotenv` (to read the API key from `.env`).
- `uv add` installs them into the project's own environment (`.venv/`), records them in `pyproject.toml`, and creates a lock file (`uv.lock`) that pins exact versions. No separate `uv sync` is needed.
- `uv run` executes a command inside that environment, here to check the versions of FastMCP, MCP, Python and the platform.
- Run the following commands from inside the mcp-demo-server directory:
```bash
uv add fastmcp requests python-dotenv
uv run fastmcp version
```

In [20]:
!uv run --directory /Users/arif/Documents/genai-course/Notebooks/mcp_demo_server fastmcp version

FastMCP version:                                                           4.1.0
MCP version:                                                               2.3.0
Python version:                                                           3.12.7
Platform:                                           macOS-27.0.1-arm64-arm-64bit
FastMCP root path: /Users/arif/Documents/genai-course/Notebooks/mcp_demo_server…

🎉 FastMCP update available: 4.1.1
Run: pip install --upgrade fastmcp


## Step 2. Write the MCP server code

**i. Store the API key in a `.env` file**
- The `get_weather` tool calls [WeatherAPI.com](https://www.weatherapi.com/), which needs an API key. Create a free account there and copy your key.
- Keep the key out of your code: put it in a `.env` file in the project folder, and add `.env` to `.gitignore` so it is never committed. The `.gitignore` that `uv init` created already excludes `.venv/`, but not `.env`.
```bash
echo "WEATHER_API_KEY=your_key_here" > .env   # Create .env with your key (replace your_key_here)
echo ".env" >> .gitignore                     # Exclude .env from git
```
> On Windows, it is safer to create `.env` in your code editor instead of with `echo`, since older PowerShell versions save the file in an encoding that `python-dotenv` may not read correctly.

**ii. Replace the contents of `main.py`**
- `uv init` created a placeholder `main.py` that only prints a greeting. Replace its entire contents with the code below.
- The server exposes the three kinds of things an MCP server can offer to a client:
  - **Tools**: functions the model can call, such as `add_numbers`, `roll_dice` and `get_weather`. Each one is marked with the `@mcp.tool()` decorator.
  - **Resources**: read-only data the client can load, here a greeting and some sample JSON.
  - **Prompts**: reusable prompt templates the user can pick in the client, here `weather_report`.
- FastMCP builds each tool's name, description and input schema from the function name, docstring and type hints, so you never write that JSON yourself. Because the docstring becomes the description the model reads, keep it clear and accurate.

```python
import random # Import the random module from Python's standard library, that provides functions to generate random numbers
import json
import datetime
import os
from pathlib import Path
import requests
from dotenv import load_dotenv
from fastmcp import FastMCP # Import the FastMCP class from the fastmcp package that is a higher-level abstraction (wrapper) for building MCP servers quickly and easily
from fastmcp.resources import TextResource # For text/plain and application/json content



# Load environment variables (API keys, etc.) from the .env file in this project folder.
# The path is built from this file's own location, so it also works when
# Claude Desktop starts the server from a different working directory.
# Variables passed in Claude Desktop's config ("env") are not overridden.
load_dotenv(Path(__file__).resolve().parent / ".env")
# Read WeatherAPI key from environment
weather_api_key = os.getenv("WEATHER_API_KEY")

# Create the server instance, giving it a name/identifier
mcp = FastMCP(name="Demo-server")


# -------------------------------------------------------------------------------------------------
# Define tools: to mark a function as a tool, we use the decorator @mcp.tool(), that is provided by FastMCP.
# Instead of writing JSON manually, you define tools using decorators.
# MCP automatically generates:
        # Tool name
        # Description (from docstring)
        # Input schema (from type hints)
        # Output schema
        # Metadata
# This becomes part of MCP's tool registry, which the model can discover and invoke dynamically.
# --------------------------------------------------------------------------------------------------

@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return their sum."""
    return a+b

@mcp.tool()
def roll_dice(n_dice: int = 1) -> list[int]:
    """Roll n_dice 6-sided dice and return the results."""
    return [random.randint(1,6) for _ in range(n_dice)] # generates a random integer between 1 and 6 (inclusive).

@mcp.tool()
def get_datetime() -> str:
    """Return the current date and time as a string."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@mcp.tool()
def fahrenheit_to_celsius(temp_f: float) -> float:
    """Convert a temperature from Fahrenheit to Celsius."""
    return (temp_f - 32) * 5.0 / 9.0

@mcp.tool()
def reverse_string(text: str) -> str:
    """Reverse the given text string."""
    return text[::-1]

@mcp.tool()
def count_words(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

@mcp.tool()
def get_weather(city:str = "Lahore") -> dict:
    """Fetch current weather info for a given city using WeatherAPI.com"""
    if not weather_api_key:
        return {"error": "Weather API key not set in .env file"}
    try:
        url = "https://api.weatherapi.com/v1/current.json"   # https, so the API key is not sent in plain text
        response = requests.get(url, params={"key": weather_api_key, "q": city}, timeout=5)
        response.raise_for_status()
        data = response.json()
        # Return structured info
        return {
            "Country": data['location']['country'],
            "City": data['location']['name'],
            "Latitude": data['location']['lat'],
            "Longitude": data['location']['lon'],
            "Temperature (C)": data['current']['temp_c'],
            "Temperature (F)": data['current']['temp_f'],
            "Condition": data['current']['condition']['text'],
            "Humidity (%)": data['current']['humidity'],
            "Wind (km/h)": data['current']['wind_kph'],
            "Feels Like (C)": data['current']['feelslike_c'],
            "Last Updated": data['current']['last_updated'],
        }
    except Exception as e:
        return {"error": str(e)}

# ---------------------------------------------------------------------
# Define Resources (static or retrievable data for MCP clients)
# ---------------------------------------------------------------------

# Static Resource 1: A static greeting message
mcp.add_resource(
    TextResource(
        uri="local://greeting",          # A unique identifier or address for the resource.
        name="greeting",                 # A human-readable name for the resource.
        description="A simple greeting message",     # Short explanation of what the resource is.
        text="Hello from your local MCP server! 🎉"  # The actual content of the resource (the text that clients will get).
    )
)

# Static Resource 2: Sample JSON data
sample_json_data = {"students": ["Hadeed", "Yashal", "Zalaid", "Miraal"], "class": "AI Basics"}
mcp.add_resource(
    TextResource(
        uri="local://sample-data",
        name="sample-data",
        description="Sample student data",
        text=json.dumps(sample_json_data, indent=2)
    )
)



# ---------------------------------------------------------------------
# Define Prompts: reusable prompt templates that the user can pick in the client
# ---------------------------------------------------------------------

@mcp.prompt()
def weather_report(city: str = "Lahore") -> str:
    """Write a short, friendly weather report for a city."""
    return f"Use the get_weather tool to fetch the current weather in {city}, then write a friendly three-sentence weather report."


if __name__ == "__main__":
    mcp.run()
```

**iii. Commit your work**
- Check that `.env` is **not** in the list of files to be committed, then commit:
```bash
git status                           # .env should not appear here
git add .
git commit -m "Add MCP demo server"
```

## Step 3. Run the Server and call its Tools via Command Line
- Run all commands below from inside the `mcp_demo_server` project folder. Prefixing them with `uv run` makes them use the project's own environment, so there is no need to activate `.venv` yourself.

**i. Check what the server offers**
- `fastmcp list` starts the server for a moment, asks it what it offers, prints the result and stops it again. It is the quickest way to catch a typo or an import error before connecting any client.
- By default it prints only the tools. Add `--resources --prompts` to also see the resources and the prompt.
```bash
uv run fastmcp list main.py --resources --prompts
```
- You should see the seven tools (`add_numbers`, `roll_dice`, `get_datetime`, `fahrenheit_to_celsius`, `reverse_string`, `count_words`, `get_weather`), the two resources (`local://greeting`, `local://sample-data`) and the `weather_report` prompt.

In [21]:
!uv run --directory /Users/arif/Documents/genai-course/Notebooks/mcp_demo_server fastmcp list main.py --resources --prompts

[10/19/26 11:37:56] INFO     Starting MCP server 'Demo-server'  ]8;id=6730751;file:///Users/arif/Documents/genai-course/Notebooks/mcp_demo_server/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=6730752;file:///Users/arif/Documents/genai-course/Notebooks/mcp_demo_server/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py#241\241]8;;\
                             with transport 'stdio'                             
Tools (7)

  add_numbers(a: float, b: float) -> dict
    Add two numbers together and returns their sum.

  roll_dice(n_dice: int = 1) -> dict
    Roll n_dice 6-sided dice and return the result

  get_datetime() -> dict
    Return the current date and time as a string.

  fahrenheit_to_celsius(temp_f: float) -> dict
    Convert a temperature from Fahrenheit to Celsius.

  reverse_string(text: str) -> dict
    Reverse the given text string.

  count_words(text: str) -> dict
    Count the number of words in a

**ii. Call a tool from the terminal**
- `fastmcp call` runs a single tool and prints its result. Arguments are passed as `name=value`.
```bash
uv run fastmcp call main.py add_numbers a=3 b=4
uv run fastmcp call main.py get_weather city=London
```
- If `get_weather` returns `"Weather API key not set in .env file"`, check that `.env` is in the project folder and contains `WEATHER_API_KEY=...` (see Step 2).

In [22]:
!uv run --directory /Users/arif/Documents/genai-course/Notebooks/mcp_demo_server fastmcp call main.py add_numbers a=3 b=4

[10/19/26 11:38:22] INFO     Starting MCP server 'Demo-server'  ]8;id=907868;file:///Users/arif/Documents/genai-course/Notebooks/mcp_demo_server/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=907869;file:///Users/arif/Documents/genai-course/Notebooks/mcp_demo_server/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py#241\241]8;;\
                             with transport 'stdio'                             
{
  "result": 7.0
}


In [23]:
!uv run --directory /Users/arif/Documents/genai-course/Notebooks/mcp_demo_server fastmcp call main.py get_weather city=Lahore

[10/19/26 11:38:45] INFO     Starting MCP server 'Demo-server'  ]8;id=13964756;file:///Users/arif/Documents/genai-course/Notebooks/mcp_demo_server/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=13964757;file:///Users/arif/Documents/genai-course/Notebooks/mcp_demo_server/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py#241\241]8;;\
                             with transport 'stdio'                             
{
  "Country": "Pakistan",
  "City": "Lahore",
  "Latitude": 31.5497,
  "Longitude": 74.3436,
  "Temperature (C)": 30.3,
  "Temperature (F)": 86.5,
  "Condition": "Sunny",
  "Humidity (%)": 28,
  "Wind (km/h)": 7.6,
  "Feels Like (C)": 28.7,
  "Last Updated": "2026-10-19 11:30"
}


<font color=purple> <h1> 2. Run Your MCP Server in Debugging/Developer Mode and test it via MCP Inspector</h1>

## a. What is MCP Inspector?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">MCP Inspector is the official web-based developer tool for MCP. It connects to your MCP server as a client, lets you call its tools, read its resources and try its prompts by hand, and shows every JSON-RPC message that goes back and forth.</div></h3>

<div style="flex: 1; text-align: center;">
    <img src="../images/mcp-inspector2.png" alt="MCP Inspector v2" style="max-width: 1500px; width: 100%; height: auto;">
</div>

The screenshot shows **MCP Inspector v2** connected to our Demo-server, right after running the `add_numbers` tool:
- **Top bar:** the server name, the four tabs **Servers · Tools · Prompts · Resources**, the connection status with its round-trip time (e.g. *Connected (1585ms)*), and buttons for settings, dark mode and the side panel.
- **Servers tab:** one card per server, showing its transport (**STDIO** or **Streamable HTTP**), the command or URL, the MCP protocol version, and a toggle to **connect / disconnect**. **Connection Info** shows the details of the current connection.
    - *Transport:* how data moves between client and server. **stdio** is used for local servers, and **Streamable HTTP** for remote ones (the older **SSE** transport is deprecated).
    - *Command and arguments:* the program that launches a stdio server and the parameters passed to it, e.g. `fastmcp run main.py`.
    - *Environment variables:* key-value pairs, such as `WEATHER_API_KEY`, passed to the server process.
    - *Authentication:* local stdio servers usually need none. Remote servers may need a bearer token or an OAuth sign-in.
- **Tools / Prompts / Resources tabs:** the left column lists what the server exposes (with search). Selecting an item opens a form generated from its input schema (or **Edit as JSON**). **Execute Tool** shows the **Results**: the text the model would see, and the **Structured Output** as JSON.
- **Side panel, with three views:**
    - **Logs:** the *Log Stream* of log messages sent by the server.
    - **Protocol:** every JSON-RPC message with its direction (*client → server*), method (`tools/list`, `prompts/list`, `tools/call`, …), latency and status. You can replay or pin a message. This is the best place to see how MCP really works.
    - **Console:** additional diagnostic messages.

- https://modelcontextprotocol.io/docs/tools/inspector
- **MCP Inspector:** https://github.com/modelcontextprotocol/inspector

## b. Installing and Running MCP Inspector
MCP Inspector **v2** needs **Node.js 22.19 or newer** (`node -v`). It opens a web page at `http://127.0.0.1:6274`, protected by a per-launch token that is printed in the terminal.
- **Option 1:**
    - Use `npm` to install it permanently, then run it in standalone mode (recommended if you use it often):
```bash
$ npm install -g @modelcontextprotocol/inspector
$ mcp-inspector
```
- **Option 2:**
    - Use `npx`, which downloads and runs it temporarily:
```python
!npx @modelcontextprotocol/inspector
```
- **Option 3 (used below):**
    - Let FastMCP start your server and the Inspector together with `fastmcp dev inspector`.

## c. Run MCP Server in Debug/Development Mode
- The following command runs the MCP server in `main.py` and automatically launches the MCP Inspector. Use it during development and debugging.
```bash
uv run --directory mcp_demo_server fastmcp dev inspector main.py
```
- In **FastMCP 4**, `dev` is a group of development tools, and the Inspector is the `inspector` subcommand: `fastmcp dev inspector main.py`. (The older form `fastmcp dev main.py` now fails with *Unknown command*.)
- The server is started with **auto-reload**: when you save a change to `main.py`, it restarts automatically.
- The terminal prints a link such as `http://127.0.0.1:6274?MCP_INSPECTOR_API_TOKEN=...`. Open it in your browser. The token is created fresh for every launch, so it is safe to show.
- On the **Servers** tab, switch on the toggle of the `fastmcp` server card to connect, then go to the **Tools** tab.
- Remember to stop the server when you are done testing: press **I, I** (interrupt the kernel) in Jupyter, or use **Kernel → Interrupt** from the menu.
- If, when you run the server again, you see an error that the port is already in use, list the process holding port 6274 and kill it with the two cells below. (`lsof -ti` prints only the process ID, which `xargs` passes to `kill`.)
```bash
!lsof -i :6274
!lsof -ti :6274 | xargs kill -9
```

In [26]:
!uv run --directory "/Users/arif/Documents/genai-course/Notebooks/mcp_demo_server" fastmcp dev inspector main.py

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏Starting MCP inspector...

MCP Inspector Web is up and running at:
   http://127.0.0.1:6274?MCP_INSPECTOR_API_TOKEN=119375931cbcff1b85838a575d8e0a4560ad224935984dd059869b8229723255

   Sandbox (MCP Apps): http://127.0.0.1:6275/sandbox

   Auth token: 119375931cbcff1b85838a575d8e0a4560ad224935984dd059869b8229723255

   Secrets: OS keychain

Opening browser...
^C
⠙

<font color=purple> <h1> 3. Run Your MCP Server Locally in Production Mode and Access it in Claude Desktop</h1></font>


## a. Run server in production mode on your Local Machine
- You can run the server inside the Jupyter notebook or in a terminal with the following commands (in the terminal, replace the path with the absolute path of **your** `mcp_demo_server` folder):
```bash
!uv run --directory mcp_demo_server fastmcp run main.py
$ uv run --directory "/Users/arif/genai-course/Notebooks/mcp_demo_server" fastmcp run main.py
```
- This is how you'd run it in production (not just testing/debugging): plain stdio, no Inspector and no auto-reload. In practice you rarely start a stdio server by hand. The **client** starts it, as Claude Desktop does in the next step.


In [18]:
!uv run --directory /Users/arif/Documents/genai-course/Notebooks/mcp_demo_server fastmcp run main.py



╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                                                              │
│                                FastMCP 4.1.0                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      Demo-server, 4.1.0                          │
│                  🚀 Deplo

## b. Accessing Demo Server from Claude Desktop via its Config File
- We want Claude Desktop to read its config file (`claude_desktop_config.json`), and then:
    - start (*spawn*) the server using `"command"` + `"args"`, and
    - talk to it over **stdio**.
- Open the file from **Settings → Developer → Edit Config** (see Lec-42 for its location and for reading the logs), and add a `Demo-server` entry **inside** the existing `mcpServers` object, next to any servers you added in Lec-42:
```json
{
  "mcpServers": {
    "Demo-server": {
      "command": "/Users/arif/.local/bin/uv",
      "args": [
        "run",
        "--directory",
        "/Users/arif/Documents/genai-course/Notebooks/mcp_demo_server",
        "fastmcp",
        "run",
        "main.py"
      ],
      "env": { "WEATHER_API_KEY": "your weather api key" }
    }
  }
}
```
- `command` is the **absolute path** of `uv` (find it with `which uv`; on Windows `where uv`). Claude Desktop does not read your shell's `PATH`, so a bare `uv` may fail with `spawn uv ENOENT`.
- `--directory` makes uv run inside the project folder with the **project's own** environment, so `fastmcp`, `requests` and `python-dotenv` are all available.
- `env` passes the API key to the server. (`main.py` also reads `keys/.env`, but a value set here takes precedence.) Values in this file are stored as plain text.
- Save the file, **fully quit** Claude Desktop (Cmd+Q on macOS, or **Quit** from the system tray on Windows) and start it again. Then click **+ → Connectors** in the chat box: the `Demo-server` tools should be listed.

> **Shortcut:** FastMCP can write the entry for you:
> ```python
> !uv run --directory mcp_demo_server fastmcp install claude-desktop main.py --name Demo-server --env WEATHER_API_KEY=your-key
> ```
> It writes `"command": "uv"` (not an absolute path) and runs `main.py` with `--with fastmcp`, plus a few extra keys set to `null` (`transport`, `type`, `cwd`, `timeout`, …), which you can ignore. If the server does not start, replace `uv` with its absolute path as shown above.

> **Practice** (approve each tool call):
> 1. *"Using the Demo-server tools, add 41.5 and 0.5."*
> 2. *"Roll 5 dice and tell me how many sixes I got."*
> 3. *"What is the weather in Lahore right now? Convert 98.6 °F to Celsius as well."*
> 4. Click **+**, choose the **Demo-server** prompt `weather_report`, enter a city, and send it.

In [ ]:
!uv run --directory mcp_demo_server fastmcp run main.py

<font color=purple> <h1> 4. Deploy your MCP Server on render.com and Access it in Claude Desktop</h1></font>

<div style="text-align:center;">
    <img src="../images/mcp-new-22.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **How it connects:** Claude → Custom connector → Render. Up to now Claude Desktop **started** our server on our own laptop and talked to it over **stdio**. A server on Render runs in the cloud and speaks **Streamable HTTP** at a public URL such as `https://<your-service>.onrender.com/mcp`, so any student can add it to Claude Desktop on their own machine.
- **What you can do once it's connected:**
    - **One URL for everyone:** every student in the class connects to the same server.
    - **Same tools, remotely:** `add_numbers`, `roll_dice`, `get_weather` and the rest work exactly as before.
    - **Auto-deploy on push:** every `git push` to GitHub builds and ships a new version.
    - **Keys stay on the server:** `WEATHER_API_KEY` lives in Render's environment variables, not on the students' machines.
- **The big picture:** ① Push the server to **GitHub** → ② Create a free **Web Service** on Render → ③ Add `https://<your-service>.onrender.com/mcp` to Claude Desktop as a **custom connector**. Steps 1 to 7 below walk through this in detail.

## a. Local vs. Remote at a Glance

| | Section 3: Local server | Section 4: Remote server on Render |
|---|---|---|
| **Who starts the server** | Claude Desktop, as a child process | Render, which keeps it running (on the free plan it sleeps when idle) |
| **Transport** | stdio | Streamable HTTP (`/mcp` endpoint) |
| **How Claude finds it** | `command` + `args` in `claude_desktop_config.json` | A URL, added as a **custom connector** |
| **Where the API keys live** | `env` in the config file / your `.env` | Render → **Environment** variables |
| **Who can use it** | Only you, on this computer | Anyone who has the URL (all students) |
| **Cost** | Free | Free instance type (sleeps after 15 minutes without traffic) |

## b. Prerequisites
- A **GitHub** account (https://github.com) and **Git** installed on your machine.
- A **Render** account. Sign up at https://dashboard.render.com (signing up with GitHub saves a step later).
- **uv** installed (section 1) and, optionally, a free **WeatherAPI** key from https://www.weatherapi.com for the `get_weather` tool.
- On the student side: **Claude Desktop** with a claude.ai account. Free accounts can add **one** custom connector, which is all we need.


## Step 1. Create the Deployment Project

**i. Creating a project from scratch**
```bash
mkdir mcp_render_server                # Create a new project directory
cd mcp_render_server                   # Navigate into the directory
uv init . --no-package --python 3.12 # Initialize the project (flat layout with main.py), pinned to Python 3.12
ls -a
.	..	.git	.gitignore	.python-version	main.py	pyproject.toml	README.md
```

**iii. Adding the dependencies**
- Our server needs `fastmcp`, `requests` (for the weather API) and `python-dotenv` (to read the API key from `.env`).
- `uv add` installs them into the project's own environment (`.venv/`), records them in `pyproject.toml`, and creates a lock file (`uv.lock`) that pins exact versions. No separate `uv sync` is needed.
- `uv run` executes a command inside that environment, here to check the versions of FastMCP, MCP, Python and the platform.
- Run the following commands from inside the mcp-demo-server directory:
```bash
uv add fastmcp requests python-dotenv
uv run fastmcp version
```

## Step 2: Write the Remote-Ready Server
The code below is almost similar to the one discussed above, with four changes as mentioned below:
1. **`load_dotenv()`** reads a local `.env` when you test on your laptop. On Render there is no `.env` file, and the keys come from Render's environment variables instead.
2. **Optional token protection:** if the environment variable `MCP_API_TOKEN` is set, every request to `/mcp` must carry the header `Authorization: Bearer <token>`, otherwise the server answers `401 Unauthorized`. If it is not set, the server is open to anyone who knows the URL.
    > For a class demo, leave `MCP_API_TOKEN` **unset**. On most Claude plans, a custom connector cannot send a fixed token yet (see Step 6). FastMCP's `StaticTokenVerifier` is also meant for development and testing, not production.
3. **A `/health` page** that simply returns `OK`. It is not protected by the token. Render can use it as a health check (Step 4), and you can open it in a browser to see whether the service is up or to wake it (Step 7).
4. **`mcp.run(transport="http", host="0.0.0.0", port=...)`**: Streamable HTTP instead of stdio, listening on all network interfaces, on the port that Render passes in the `PORT` environment variable (Render's default is `10000`). On your laptop, where `PORT` is not set, it falls back to `8000`.

```python
$ cat ./mcp_render_server/main.py
import random, datetime, os
import requests
from dotenv import load_dotenv
from fastmcp import FastMCP
from fastmcp.resources import TextResource
from fastmcp.server.auth.providers.jwt import StaticTokenVerifier
from starlette.requests import Request
from starlette.responses import PlainTextResponse

# On your laptop the keys come from a local .env file; on Render they come from the
# service's Environment Variables (a missing .env file is simply ignored).
load_dotenv()
weather_api_key = os.getenv("WEATHER_API_KEY")

# Optional: if MCP_API_TOKEN is set, every request must send "Authorization: Bearer <token>"
api_token = os.getenv("MCP_API_TOKEN")
auth = StaticTokenVerifier(tokens={api_token: {"client_id": "student"}}) if api_token else None

mcp = FastMCP(name="Demo-server", auth=auth)

@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return their sum."""
    return a + b

@mcp.tool()
def roll_dice(n_dice: int = 1) -> list[int]:
    """Roll n_dice 6-sided dice and return the result."""
    return [random.randint(1, 6) for _ in range(n_dice)]

@mcp.tool()
def get_datetime() -> str:
    """Return the server's current date and time (UTC on Render) as a string."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@mcp.tool()
def fahrenheit_to_celsius(temp_f: float) -> float:
    """Convert a temperature from Fahrenheit to Celsius."""
    return (temp_f - 32) * 5.0 / 9.0

@mcp.tool()
def reverse_string(text: str) -> str:
    """Reverse the given text string."""
    return text[::-1]

@mcp.tool()
def count_words(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

@mcp.tool()
def get_weather(city: str = "Lahore") -> dict:
    """Fetch current weather info for a given city using WeatherAPI.com"""
    if not weather_api_key:
        return {"error": "WEATHER_API_KEY is not set on the server"}
    try:
        url = "https://api.weatherapi.com/v1/current.json"
        response = requests.get(url, params={"key": weather_api_key, "q": city}, timeout=5)
        response.raise_for_status()
        data = response.json()
        return {
            "Country": data['location']['country'],
            "City": data['location']['name'],
            "Temperature (C)": data['current']['temp_c'],
            "Condition": data['current']['condition']['text'],
            "Humidity (%)": data['current']['humidity'],
            "Wind (km/h)": data['current']['wind_kph'],
            "Last Updated": data['current']['last_updated'],
        }
    except Exception as e:
        return {"error": str(e)}

mcp.add_resource(TextResource(uri="local://greeting", name="greeting",
                              description="A simple greeting message",
                              text="Hello from your remote MCP server on Render! 🎉"))

# A plain web page to check that the service is up (and to wake it after a spin-down).
# Custom routes are not protected by the token.
@mcp.custom_route("/health", methods=["GET"])
async def health(request: Request) -> PlainTextResponse:
    return PlainTextResponse("OK")

if __name__ == "__main__":
    # Render tells the app which port to use through the PORT variable (default 10000).
    port = int(os.getenv("PORT", 8000))
    mcp.run(transport="http", host="0.0.0.0", port=port)
```

**Test it on your laptop first.** Start the server in a **terminal** (not a notebook cell, because it keeps running):
```bash
uv run --directory mcp_render_server main.py
```
- Then, open a browser and go to http://127.0.0.1:8000/health, which will return `OK`
- Then run the following command in another terminal (in the same directory) that will list what the server offers:
```bash
uv run fastmcp list http://127.0.0.1:8000/mcp --auth none --resources
```
- `--auth none` is needed because the FastMCP CLI tries an OAuth sign-in by default when it talks to a URL.
- Press `Ctrl+C` in the first terminal to stop the server.

## Step 3: Push the Project to GitHub
1. On GitHub, click **New repository**, name it `mcp_render_server`, and leave it **empty** (no README, no `.gitignore`, no license), so the first push does not conflict.
2. Run these commands in a **terminal** from the folder that contains `mcp_render_server` (a notebook cell cannot answer GitHub's sign-in prompt). Replace `<your-username>`:
```bash
cd mcp_render_server
git init                              # Safe if uv already created the repository; needed if the folder sits inside another Git repository
git status                            # Check: .env and .venv/ must NOT be listed
git add .
git commit -m "MCP server for Render"
git branch -M main
git remote add origin https://github.com/<your-username>/mcp_render_server.git
git push -u origin main
```
- Make sure `main.py`, `pyproject.toml`, `uv.lock` and `.python-version` are all in the root of the repository. Render needs `uv.lock` there to use uv.


## Step 4: Create a Web Service on Render
1. In the Render dashboard, click **New** › **Web Service** and connect your GitHub account if asked. Pick the `mcp_render_server` repository.
2. Fill in the form:

| Setting | Value |
|---|---|
| **Name** | e.g. `mcp-demo-server` (this becomes `https://mcp-demo-server.onrender.com`, plus a random suffix if the name is taken) |
| **Language** | `Python 3` |
| **Branch** | `main` |
| **Root Directory** | leave empty |
| **Build Command** | `uv sync --frozen && uv cache prune --ci` |
| **Start Command** | `uv run main.py` |
| **Instance Type** | `Free` |

- `uv sync --frozen` installs exactly the versions in `uv.lock` and fails if the lock file is out of date. `uv cache prune --ci` keeps the build small.
3. Under **Environment Variables**, add `WEATHER_API_KEY` with your key. Only add `MCP_API_TOKEN` if you want token protection (read Step 6 first).
4. Click **Deploy Web Service** and watch the logs. The deploy is live when you see `Uvicorn running on http://0.0.0.0:10000` and the status turns to **Live**.
5. Optional: under **Settings** › **Health Checks**, set the path to `/health`. Render then checks that page and only switches traffic to a new deploy once it answers `OK`.


## Step 5: Test the Deployed Server via your Terminal
- Open `https://<your-service>.onrender.com/health` in a browser. It should show `OK`.
- From your laptop terminal (go to the mcp-render-server directory), and list the tools on the live server and call one:
```bash
uv run  fastmcp list https://mcp-render-server-xzbh.onrender.com/mcp --auth none --resources 
uv run fastmcp call https://mcp-render-server-xzbh.onrender.com/mcp get_weather city=Lahore --auth none
```
- If you set `MCP_API_TOKEN`, replace `--auth none` with `--auth <your-token>`. Pass the token on its own: the CLI adds the `Bearer` prefix itself.

## Step 6: Test the Deployed Server via Claude Desktop 
**Option 1:**
-  In Claude Desktop go to **Customize** › **Connectors** and click **Add custom connector**.
-  Enter a name (e.g. `demo-server-remote`) and the URL `https://mcp-render-server-xzbh.onrender.com/mcp`
-  If the dialog asks about authentication, choose **No sign-in**. Then click **Add**.

**Option 2:**
- Edit the `claude_desktop_config.json` and add the following, then restart the Claude Desktop app:
```json
"demo-server-remote": {
  "command": "npx",
  "args": [
    "-y",
    "mcp-remote",
    "https://mcp-render-server-xzbh.onrender.com/mcp",
    "--header",
    "Authorization:${AUTH_HEADER}"
  ],
  "env": {
    "PATH": "/Users/arif/.nvm/versions/node/v22.22.3/bin:/usr/local/bin:/usr/bin:/bin",
    "AUTH_HEADER": "Bearer <the MCP_API_TOKEN value you set on Render>"
  }
}
```

> **With a token (`MCP_API_TOKEN` set):** a custom connector can only send the token if your account has the **Request headers** option in the **Add custom connector** dialog. That option is in beta and only available to some organizations. If you have it, choose **No sign-in**, add a header named `authorization`, and enter the value as `Bearer <your-token>`, including the word `Bearer` and the space.


> **Practice:** In a chat, click **+** › **Connectors** and make sure `demo-server-remote` is switched on.
> 1. *"Using the demo-server-remote tools, add 12.5 and 30.75."*
> 2. *"Roll 4 dice and tell me the total."*
> 3. *"What is the weather in Lahore and in Karachi right now? Compare them in a table."*
> 4. *"What date and time does the server report?"* → notice it is **UTC**, because the Render machine runs on UTC, not Pakistan time.
> 5. *"Reverse the string 'Model Context Protocol' and count its words."*

> **Living with the Free Plan, and Updating the Server**
- **Spin-down:** a free web service goes to sleep after **15 minutes** without incoming traffic. The next request wakes it, which takes about **one minute**. A tool call from Claude during that minute may time out. Before class, open `https://<your-service>.onrender.com/health` in a browser and wait for `OK`, then use the connector.
- **Monthly hours:** each Render workspace gets **750 free instance hours** per month. A sleeping service uses none.
- **Updating:** change `main.py`, then `git add`, `git commit` and `git push`. Render rebuilds and redeploys automatically. Claude picks up new or changed tools the next time it connects. If a new tool doesn't appear, toggle the connector off and on in **+** › **Connectors**.
- **Changing a token:** a custom connector's authentication settings can't be edited. To change the token, remove the connector and add it again.

## Update, Monitor and Troubleshoot
- **Update the server:** edit `main.py` → `git commit -am "Add a new tool"` → `git push`. Render rebuilds and redeploys automatically. Start a new chat in Claude (or reconnect the connector) to see new tools.
- **Monitor:** the **Logs** tab shows every request and every error. The **Events** tab shows deploys.

| Symptom | Likely cause | Fix |
|---|---|---|
| The first request after a break takes about a minute, or Claude reports a timeout | A **Free** instance spins down after **15 minutes** without traffic | Open `https://<your-service>.onrender.com/health` in a browser to wake it, then retry |
| Deploy fails with *"No open ports detected"* | The server listens on `127.0.0.1` or on a fixed port | Use `host="0.0.0.0"` and `port=int(os.getenv("PORT"))`, as in step 2 |
| Build fails at `uv sync --locked` | `uv.lock` does not match `pyproject.toml` | Run `uv lock` locally, then commit and push |
| Build log does not use uv / `uv: command not found` | `uv.lock` is not in the repo root (or Root Directory is wrong) | Commit `uv.lock` to the root, or fix **Root Directory**. Fallback: `uv export --no-hashes --format requirements-txt > requirements.txt`, Build Command `pip install -r requirements.txt`, Start Command `python main.py` |
| `get_weather` returns *"WEATHER_API_KEY is not set"* | The environment variable is missing | Render → **Environment** → add it → **Save, rebuild, and deploy** |
| Connector shows an error / *401 Unauthorized* | Token missing or wrong | The header must be exactly `Authorization: Bearer <token>`, with the same token as `MCP_API_TOKEN` |
| Browser shows a JSON error at `/mcp` | Normal: the endpoint only answers MCP clients | Use `/health` to check the service |

**Free-tier facts worth knowing:** each workspace gets **750 free instance hours per month** (a sleeping service uses none); there is **no persistent disk**, so files written by the server disappear on every redeploy, restart or spin-down; and Render may restart a free service at any time.

> **Security note:** A public MCP server runs **your** code with **your** API keys for anyone who knows the URL. Set `MCP_API_TOKEN` before sharing the URL, keep `.env` out of Git, and never deploy tools that touch the server's files or shell. The static token used here (`StaticTokenVerifier`) is fine for a class demo. A production server should use OAuth (FastMCP has ready-made providers for GitHub, Google and others), which also gives Claude the **Sign in** flow you saw with directory connectors.

📖 References: [Render: Web Services](https://render.com/docs/web-services) · [Render: Free instances](https://render.com/docs/free) · [Render: Python version](https://render.com/docs/python-version) · [FastMCP: HTTP deployment](https://gofastmcp.com/deployment/http) · [Claude: custom connectors](https://support.claude.com/en/articles/11175166-getting-started-with-custom-connectors-using-remote-mcp) · [mcp-remote](https://github.com/geelen/mcp-remote)

<font color=purple> <h1> 5. Access a Remote Flight-Search MCP Server from Claude Desktop</h1></font>

### https://flight-mcp-server.onrender.com/mcp

<div style="text-align:center;">
    <img src="../images/mcp-new-25.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **How it connects:** Claude → Custom connector → Flight server. In section 4 you deployed **your own** server. Here you only **use** a server that is already running on Render at `https://flight-mcp-server.onrender.com/mcp`. You write no code: you add the URL to Claude Desktop, and Claude calls the server's tool whenever a question needs live flight prices. The server then calls the **flightapi.io** REST API with its own API key, which never leaves Render.
- **What you can do once it's connected:**
    - **`search_flights`:** one tool that takes a route, a date and a cabin class.
    - **Any two cities:** you name the cities, and Claude finds their IATA airport codes.
    - **Live fares in USD:** airline and flight numbers, departure and arrival times, stops, and the booking site that offers the price.
    - **One-way, one adult:** Economy or Business, cheapest first.
- **Setting it up in 3 steps:** ① Wake the server → ② Add it as a custom connector → ③ Ask Claude for flights.

## The Server at a Glance

| | |
|---|---|
| **URL** | `https://flight-mcp-server.onrender.com/mcp` |
| **Transport** | Streamable HTTP |
| **Authentication** | None, the server is open |
| **Tool** | `search_flights(dep_iata, arr_iata, date, cabin="Economy")` |
| **Data source** | flightapi.io, live fares |
| **Returns** | Up to **10** one-way itineraries, cheapest first: airlines and flight numbers, departure, arrival, stops, duration in minutes, price in USD, booking site |
| **Limits** | One-way, 1 adult, prices in USD. All students share **one** free flightapi.io quota of **100 searches per 30 days**, so don't repeat the same search again and again |

| Parameter | Meaning | Example |
|---|---|---|
| `dep_iata` | 3-letter IATA code of the departure airport | `LHE` (Lahore) |
| `arr_iata` | 3-letter IATA code of the arrival airport | `DXB` (Dubai) |
| `date` | Travel date, `YYYY-MM-DD` | `2026-11-20` |
| `cabin` | `Economy` or `Business` (default `Economy`) | `Economy` |

> You never type these codes yourself. You ask in plain English (*"Lahore to Dubai next Friday"*), and Claude works out the airport codes and the date, then fills in the parameters. That is what a tool's **input schema** is for.

## Step 1: Wake the Server and Access it via Terminal
A **Free** Render instance goes to sleep after **15 minutes** without traffic, and the first request after that takes about 30 to 60 seconds. Wake it before you connect by opening https://flight-mcp-server.onrender.com/mcp in your browser. After about 30 seconds you see **Not Found**. That is expected: the server has no web page, only the `/mcp` endpoint, but the page shows that it is **awake**.
- From the `Notebooks` folder, ask the server what it offers and it should print one tool: `search_flights(dep_iata, arr_iata, date, cabin='Economy')`, and then call the tool using following commands on the terminal:
```bash
$ uv run  fastmcp list https://flight-mcp-server.onrender.com/mcp --auth none --resources 
$ uv run fastmcp call https://flight-mcp-server.onrender.com/mcp --auth none search_flights arr_iata=SKT dep_iata=DXB date=2026-09-21 cabin=Economy
```

In [5]:
!uv run fastmcp list https://flight-mcp-server.onrender.com/mcp --auth none --resources 

Tools (1)

  search_flights(dep_iata: str, arr_iata: str, date: str, cabin: str = 
'Economy')
    
Search REAL live flight prices for a specific route and date.

Args:
    dep_iata: 3-letter IATA code of the departure airport, e.g. "SKT"
    arr_iata: 3-letter IATA code of the arrival airport, e.g. "DXB"
    date: travel date in YYYY-MM-DD format, e.g. "2026-07-21"
    cabin: "Economy" or "Business"


Resources (0)

  No resources found.



In [4]:
!uv run fastmcp call https://flight-mcp-server.onrender.com/mcp --auth none search_flights arr_iata=SKT dep_iata=DXB date=2026-11-21 cabin=Economy

{
  "ok": false,
  "error": "FlightAPI returned 403: {\"message\":\"Your request quota has 
reached its maximum limits. Kindly upgrade your plan to continue using 
it.\",\"help\":\"If you need more FREE credits before upgrading then email us at
info@flightapi.io"
}


## Step 2: Access the Server via Claude Desktop
**Option 1 (as a Custom Connector):** 
1. In Claude Desktop open **Settings → Connectors** (in newer versions **Customize → Connectors**) and choose **Add custom connector** (**+ Add → Add custom connector**).
2. **Name:** `Flight-server`. **Remote MCP server URL:** `https://flight-mcp-server.onrender.com/mcp`. Copy the URL exactly, including `https://` and the `/mcp` at the end.
3. **Authentication:** choose **No sign in**. This server is open, so no request header is needed.
4. Click **Add**. The connector should show **Connected**. Click it to see its single tool, `search_flights`.
5. In a chat, click **+ → Connectors** and make sure **Flight-server** is switched on.
> **Free plan: one custom connector.** If you already added **Demo-server (Render)** in section 4, remove it first (**Settings → Connectors → ⋯ → Remove**), or keep it and connect the flight server through the config file instead, the way section 6 does for the Booking server.


**Option 2 (edit the config file):**
- Edit the `claude_desktop_config.json` and add the following, then restart the Claude Desktop app:
```json
"flight-server": {
       "command": "npx",
       "args": ["-y", "mcp-remote", "https://flight-mcp-server.onrender.com/mcp"]
    }
```
- `command`: the program Claude Desktop starts. Replace `npx` with the absolute path from step 1 if needed.
- `-y`: lets `npx` download `mcp-remote` the first time without asking for confirmation. There is no terminal in which to type *yes*.
- `mcp-remote` + the URL: the bridge, and the remote server it forwards to.
- What gets downloaded locally: Only mcp-remote, a small Node proxy from npm, that lands in npm’s npx cache, typically /Users/arif/.npm/_npx/<hash>/node_modules/mcp-remote. If it authenticates via OAuth, tokens are stored in /Users/arif/.mcp-auth/.
- The actual MCP server donot live on your machine, rather is hosted on Render at the mentioned URL. mcp-remote just translates between Claude Desktop’s stdio and the remote HTTP endpoint.
- When it runs: The proxy starts with Claude Desktop and connects to the Render URL. If the Render service is on a free tier, it may be asleep, so the first connection can be slow or time out.

## Step 3: Ask Claude for Flights
Start a **new chat**, check that **Flight-server** is switched on under **+ → Connectors**, and try these prompts. The first time, Claude asks for permission to use the tool: click **Allow**.

> **Practice:**
> 1. *"Find the cheapest economy flight from Lahore to Dubai on 20 November 2026."* → Claude calls `search_flights(dep_iata="LHE", arr_iata="DXB", date="2026-11-20", cabin="Economy")`. Expand the tool call in the chat to see the arguments Claude chose and the JSON the server sent back.
> 2. *"What business-class flights are there from Islamabad to London on 1 December 2026? Show the airline, departure time, number of stops and price in a table."*
> 3. *"I want to fly from Karachi to Istanbul next Friday. Which option has the fewest stops, and which is the cheapest?"* → notice that Claude turns *next Friday* into a real date.

**What to observe:**
- You type city names and ordinary dates. **Claude** fills in the tool's parameters (`LHE`, `DXB`, `2026-11-20`) using the parameter descriptions the server published.
- Claude does not invent prices. It reads them from the tool result and only formats them as a table or a summary.
- Each prompt costs at least one search from the shared quota, so try each one once.

<font color=purple> <h1> 6. Access a Remote Hotel-Search MCP Server from Claude Desktop</h1></font>

## https://booking-mcp-server.onrender.com/mcp

<div style="text-align:center;">
    <img src="../images/mcp-new-26.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **How it connects:** Claude → Config file (`mcp-remote`) → Booking server. The second server runs on Render at `https://booking-mcp-server.onrender.com/mcp` and searches **Booking.com** (through RapidAPI) with its own API key. This time we connect through the **config file** from section 3, using a small bridge program called **`mcp-remote`**. There are two reasons to use a second method:
    - It does **not** use the free plan's single custom-connector slot, so you can use the Flight server and the Booking server **at the same time**.
    - It shows how a client that only speaks **stdio** can still reach a **remote** server.
- **What you can do once it's connected:**
    - **`search_hotels`:** one tool that takes a city, a check-in date and a check-out date.
    - **Guests:** the number of adults (2 if you don't say).
    - **Live prices in USD:** the 8 cheapest hotels for your dates.
    - **Star class:** each hotel's name, price and star rating.
- **Setting it up in 3 steps:** ① Open the config file → ② Paste the `booking-server` block → ③ Fully quit and restart Claude Desktop.

## The Server at a Glance

| | |
|---|---|
| **URL** | `https://booking-mcp-server.onrender.com/mcp` |
| **Transport** | Streamable HTTP |
| **Authentication** | None, the server is open |
| **Tool** | `search_hotels(city, arrival_date, departure_date, adults=2)` |
| **Data source** | Booking.com, live, through RapidAPI |
| **Returns** | Up to **8** hotels, cheapest first: name, price in **USD** for the whole stay, star class, review score and review count |
| **Limits** | Prices are in USD. All students share **one** RapidAPI quota, so don't repeat the same search again and again |

| Parameter | Meaning | Example |
|---|---|---|
| `city` | The city to search | `Dubai` |
| `arrival_date` | Check-in date, `YYYY-MM-DD` | `2026-11-20` |
| `departure_date` | Check-out date, `YYYY-MM-DD` | `2026-11-22` |
| `adults` | Number of adult guests (default `2`) | `2` |


## Step 1: Wake the Server and Access it via Terminal
A **Free** Render instance goes to sleep after **15 minutes** without traffic, and the first request after that takes about 30 to 60 seconds. Wake it before you connect by opening https://flight-mcp-server.onrender.com/mcp in your browser. After about 30 seconds you see **Not Found**. That is expected: the server has no web page, only the `/mcp` endpoint, but the page shows that it is **awake**.
- From the `Notebooks` folder, ask the server what it offers and it should print one tool: ` search_hotels(city: str, arrival_date: str, departure_date: str, adults: int =
2)`, and then call the tool using following commands on the terminal:
```bash
$ uv run fastmcp list https://booking-mcp-server.onrender.com/mcp --auth none --resources
$ uv run fastmcp call https://booking-mcp-server.onrender.com/mcp  --auth none hotels city=Lahore arrival_date=2026-10-21 departure_date=2026-10-23 adults=2
```

In [7]:
!uv run fastmcp list https://booking-mcp-server.onrender.com/mcp --auth none --resources

Tools (1)

  search_hotels(city: str, arrival_date: str, departure_date: str, adults: int =
2)
    
Search REAL live hotel prices for a city and date range.

Args:
    city: the city to search, e.g. "Lahore"
    arrival_date: check-in date in YYYY-MM-DD format
    departure_date: check-out date in YYYY-MM-DD format
    adults: number of adult guests


Resources (0)

  No resources found.



In [9]:
!uv run fastmcp call https://booking-mcp-server.onrender.com/mcp  --auth none search_hotels city=Lahore arrival_date=2026-10-21 departure_date=2026-10-23 adults=2

{
  "ok": true,
  "city": "Lahore",
  "check_in": "2026-10-21",
  "check_out": "2026-10-23",
  "total_found": 20,
  "hotels": [
    {
      "name": "Hotel Swiss Pearl",
      "price": 23.3822975907148,
      "currency": "USD",
      "rating": 5.8,
      "review_count": 377,
      "stars": 2
    },
    {
      "name": "Hotel Holiday",
      "price": 29.3326313862013,
      "currency": "USD",
      "rating": 5.3,
      "review_count": 330,
      "stars": 3
    },
    {
      "name": "Hotel The Leschanze",
      "price": 35.618195254673,
      "currency": "USD",
      "rating": 7.2,
      "review_count": 551,
      "stars": 2
    },
    {
      "name": "Hotel Grand Pakeeza",
      "price": 36.1210403641508,
      "currency": "USD",
      "rating": 8.1,
      "review_count": 802,
      "stars": 3
    },
    {
      "name": "Saffron D'or Hotels",
      "price": 43.5799094880705,
      "currency": "USD",
      "rating": 6.9,
      "review_count": 127,
      "stars": 3
    },
    {
      "nam

## Step 2: Access the Server via Claude Desktop
**Option 1 (as a Custom Connector):** 
1. In Claude Desktop open **Settings → Connectors** (in newer versions **Customize → Connectors**) and choose **Add custom connector** (**+ Add → Add custom connector**).
2. **Name:** `booking-server`. **Remote MCP server URL:** `https://booking-mcp-server.onrender.com/mcp`. Copy the URL exactly, including `https://` and the `/mcp` at the end.
3. **Authentication:** choose **No sign in**. This server is open, so no request header is needed.
4. Click **Add**. The connector should show **Connected**. Click it to see its single tool, `search_hotels`.
5. In a chat, click **+ → Connectors** and make sure **booking-server** is switched on.
> **Free plan: one custom connector.** If you already added **Demo-server (Render)** in section 4, remove it first (**Settings → Connectors → ⋯ → Remove**), or keep it and connect the booking server through the config file instead.


**Option 2 (edit the config file):**
- Edit the `claude_desktop_config.json` and add the following, then restart the Claude Desktop app:
```json
"mcpServers": {
    "booking-server": {
      "command": "npx",
      "args": ["-y", "mcp-remote", "https://booking-mcp-server.onrender.com/mcp"]
    }
  }
```
- `command`: the program Claude Desktop starts. Replace `npx` with the absolute path from step 1 if needed.
- `-y`: lets `npx` download `mcp-remote` the first time without asking for confirmation. There is no terminal in which to type *yes*.
- `mcp-remote` + the URL: the bridge, and the remote server it forwards to.
- What gets downloaded locally: Only mcp-remote, a small Node proxy from npm, that lands in npm’s npx cache, typically /Users/arif/.npm/_npx/<hash>/node_modules/mcp-remote. If it authenticates via OAuth, tokens are stored in /Users/arif/.mcp-auth/.
- The actual MCP server donot live on your machine, rather is hosted on Render at the mentioned URL. mcp-remote just translates between Claude Desktop’s stdio and the remote HTTP endpoint.
- When it runs: The proxy starts with Claude Desktop and connects to the Render URL. If the Render service is on a free tier, it may be asleep, so the first connection can be slow or time out.

## Step 3: Ask Claude for Hotels

- Save the file and **fully quit** Claude Desktop (Cmd+Q on macOS, or **Quit** from the system tray on Windows), then start it again. The first start takes a few seconds longer while `npx` downloads `mcp-remote`.
- Open **Settings → Developer**. `booking-server` should show **running**. If it shows **failed**, click **Open Logs**, which opens `mcp-server-booking-server.log`
- In a chat, click **+ → Connectors**. `booking-server` is listed with its tool `search_hotels`, and its switch is **on**.
- Try these prompts (approve each tool call):

> **Practice:**
> 1. *"Find hotels in Dubai for 2 adults from 20 to 22 November 2026, cheapest first."* → Claude calls `search_hotels(city="Dubai", arrival_date="2026-11-20", departure_date="2026-11-22", adults=2)`. Expand the tool call to see the arguments and the JSON result.
> 2. *"I need a 4-star hotel in Istanbul for 1 adult, checking in on 5 December and staying 3 nights. What are my options?"* → Claude works out the check-out date (8 December) and picks the 4-star hotels from the result.
> 3. *"For 2 adults on the weekend of 12 to 14 December, which city is cheaper per night: Bangkok or Kuala Lumpur?"* → two tool calls, then Claude divides each price by the number of nights.